## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

created an agent to help me find an apartment in austria and send me an email with the details

In [36]:
# The imports

from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, WebSearchTool, gen_trace_id
from agents.model_settings import ModelSettings
from openai.types.responses import ResponseTextDeltaEvent
from pydantic import BaseModel, Field
from sendgrid.helpers.mail import Mail, Email, To, Content
from typing import Dict
from IPython.display import display, Markdown
import sendgrid
import asyncio
import os


In [37]:
# The usual starting point

load_dotenv(override=True)


True

In [ ]:
@function_tool
def send_email_tool(content: str) -> str:
    message = Mail(
        from_email=os.getenv("EMAIL_FROM"),
        to_emails=os.getenv("EMAIL_TO"),
        subject="🏡 Innsbruck Accommodation Matches",
        html_content=content
    )

    try:
        sg = SendGridAPIClient(os.getenv("SENDGRID_API_KEY"))
        sg.send(message)
        return "✅ Email sent successfully"
    except Exception as e:
        return f"❌ Email failed: {str(e)}"

print("Email tool ready")

Email tool ready


In [39]:
print("WebSearchTool:", WebSearchTool)
print("send_email_tool:", send_email_tool)

WebSearchTool: <class 'agents.tool.WebSearchTool'>
send_email_tool: FunctionTool(name='send_email_tool', description='', params_json_schema={'properties': {'content': {'title': 'Content', 'type': 'string'}}, 'required': ['content'], 'title': 'send_email_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x0000026E317A1B20>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)


In [40]:
instructions = """
You are an accommodation assistant.

Goal:
Find shared accommodation for students in Innsbruck and nearby villages
within 20 minutes by bus.

Steps:
1. Search the web for listings
2. Focus on:
   - shared apartments / WG
   - student housing
   - Innsbruck + nearby villages
3. Summarize:
   - price
   - location
   - link
4. Keep it concise and useful
5. Then send results via email using the email tool

Only send email when results are ready.
"""

In [43]:
agent = Agent(
    name="Accommodation Finder",
    instructions=instructions,
    tools=[WebSearchTool(), send_email_tool]
)

print("✅ Agent created")

✅ Agent created


In [ ]:
with trace("Accommodation Search"):
    result = await Runner.run(
        agent,
        "Find 3–5 shared student accommodations in Innsbruck or nearby villages within 20 minutes commute and email me"
    )



There was a technical issue sending the email, but here is your concise summary of shared student accommodation options in Innsbruck and nearby villages within a 20-minute commute. You can easily copy and keep this information:

---

**1. WG Innsbruck – Harterhofweg 76a, 6020 Innsbruck**
- Shared apartment, furnished single rooms (11–19 m²), kitchen, rooftop garden.
- Commute: 4–20 min by bus to university centers.
- Contact & Info: [wg-innsbruck.com](https://wg-innsbruck.com/) | info@wg-innsbruck.com

**2. Studentenheim Innsbruck (Wilten district)**
- Furnished single rooms (15–20 m²), rent €380 all-inclusive, central.
- Commute: 5 min walk to main universities.
- Contact & Info: [studentenheim-innsbruck.at](https://www.studentenheim-innsbruck.at/) | Zollerstraße 3, 6020 Innsbruck

**3. OeAD-Guesthouse “GreenINN” – Karmelitergasse 9**
- Modern student residence, shared/single rooms, short bus ride to city center/universities (~15 min).
- Pricing: Semester bookings, €20/night for guest

In [45]:
print(result)

RunResult:
- Last agent: Agent(name="Accommodation Finder", ...)
- Final output (str):
    There was a technical issue sending the email, but here is your concise summary of shared student accommodation options in Innsbruck and nearby villages within a 20-minute commute. You can easily copy and keep this information:
    
    ---
    
    **1. WG Innsbruck – Harterhofweg 76a, 6020 Innsbruck**
    - Shared apartment, furnished single rooms (11–19 m²), kitchen, rooftop garden.
    - Commute: 4–20 min by bus to university centers.
    - Contact & Info: [wg-innsbruck.com](https://wg-innsbruck.com/) | info@wg-innsbruck.com
    
    **2. Studentenheim Innsbruck (Wilten district)**
    - Furnished single rooms (15–20 m²), rent €380 all-inclusive, central.
    - Commute: 5 min walk to main universities.
    - Contact & Info: [studentenheim-innsbruck.at](https://www.studentenheim-innsbruck.at/) | Zollerstraße 3, 6020 Innsbruck
    
    **3. OeAD-Guesthouse “GreenINN” – Karmelitergasse 9**
    -

In [46]:
result = await Runner.run(
        agent,
        "Find cheaper options under €600 and email me"
    )


print(result.final_output)

It appears there was an error sending the email. Could you please provide your email address or confirm if you’d like the results sent to a specific address? This will help me complete your request as soon as the email system is available. Meanwhile, here is the summary of shared accommodation options under €600 in Innsbruck and nearby villages:

---

### Top Options Under €600

1. **Home4students Euregio‑Campus Dormitory**
   - Price: €480–500/month (single room, all-inclusive)
   - Location: Innsbruck, near universities, very good bus access
   - [Details & Apply](https://www.home4students.at/en/our-dormitories/dormitories-innsbruck/dorm-euregio-campus/)

2. **General Shared Flats (WG)**
   - Price: €450–500/month on average, up to €600 for some central rooms
   - Location: Innsbruck, all neighborhoods, check listings for bus time
   - [Search Listings](https://lystio.at/en/rent/room/tyrol/innsbruck)

3. **Other Dorms by Home4students**
   - Price: €395–522/month (all-inclusive), var

In [ ]:
def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("mugao.dev@gmail.com")  # Change to your verified sender
    to_email = To("mugaojoy@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

In [ ]:
INSTRUCTIONS = "You are a research assistant. Given a search term, you search the web for that term and \
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 \
words. Capture the main points. Write succintly, no need to have complete sentences or good \
grammar. This will be consumed by someone synthesizing a report, so it's vital you capture the \
essence and ignore any fluff. Do not include any additional commentary other than the summary itself."

search_agent = Agent(
    name="Search agent",
    instructions=INSTRUCTIONS,
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

In [12]:
message = "Latest AI Agent frameworks in 2025"

with trace("Search"):
    result = await Runner.run(search_agent, message)

display(Markdown(result.final_output))

In 2025, several AI agent frameworks have emerged, each offering unique capabilities:

- **Nvidia's NemoClaw**: A platform enabling enterprise users to deploy autonomous AI agents, emphasizing security and open-source accessibility. ([techradar.com](https://www.techradar.com/pro/nvidia-might-be-about-to-reimagine-ai-agents-at-work-with-new-nemoclaw-release?utm_source=openai))

- **Agent Lightning**: A flexible framework for training AI agents using reinforcement learning, compatible with various existing agents with minimal code changes. ([arxiv.org](https://arxiv.org/abs/2508.03680?utm_source=openai))

- **Project Mariner**: Developed by Google DeepMind, this research prototype automates web-based tasks like shopping and information retrieval, enhancing user productivity. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Project_Mariner?utm_source=openai))

- **Google Antigravity**: An AI-powered integrated development environment (IDE) that allows developers to delegate complex coding tasks to autonomous AI agents. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Google_Antigravity?utm_source=openai))

- **GoalfyMax**: A protocol-driven multi-agent system designed for intelligent experience entities, focusing on end-to-end multi-agent collaboration. ([arxiv.org](https://arxiv.org/abs/2507.09497?utm_source=openai))

- **Cognitive Kernel-Pro**: An open-source framework for training deep research agents and agent foundation models, aiming to democratize AI agent development. ([arxiv.org](https://arxiv.org/abs/2508.00414?utm_source=openai))

- **Agent2Agent (A2A)**: An open protocol for communication between AI agents, facilitating interoperability across different systems. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Agent2Agent?utm_source=openai))

- **OpenClaw**: An open-source autonomous AI assistant software that executes tasks via large language models, using messaging platforms as its main user interface. ([en.wikipedia.org](https://en.wikipedia.org/wiki/OpenClaw?utm_source=openai))

- **Manus**: An autonomous AI agent developed by Butterfly Effect Pte Ltd, capable of independently carrying out complex real-world tasks without direct human guidance. ([en.wikipedia.org](https://en.wikipedia.org/wiki/Manus_%28AI_agent%29?utm_source=openai))

These frameworks reflect the rapid advancements in AI agent technology, offering diverse solutions for automation, development, and interoperability. 

Exception in thread Thread-7 (_run):
Traceback (most recent call last):
  File "C:\Users\PC\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\PC\Desktop\AI-ENGINEERING\agents\.venv\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\PC\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\PC\Desktop\AI-ENGINEERING\agents\.venv\Lib\site-packages\agents\tracing\processors.py", line 263, in _run
    self._export_batches(force=False)
  File "c:\Users\PC\Desktop\AI-ENGINEERING\agents\.venv\Lib\site-packages\agents\tracing\processors.py", line 295, in _export_batches
    self._exporter.export(items_to_export)
  File "c:\Users\PC\Desktop\AI-ENGINEERING\agents\.venv\Lib\site-packages\agents\tracing\processors.py", line 120, in export
    

In [7]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="openai/gpt-4o-mini")

In [ ]:
# Run the joke with Runner.run(agent, prompt) then print final_output

with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

## Now go and look at the trace

https://platform.openai.com/traces